In [ ]:
# SPARCS 2023 sepsis benchmarking starter
# Risk-adjusted in-hospital mortality, O/E by hospital, funnel plot.
#
# uv add pandas pyarrow numpy scipy scikit-learn lightgbm matplotlib
# Run cell by cell in VS Code / Jupyter with the .venv uv creates.
#
# Column names follow the Socrata API (lowercase_with_underscores); see
# DATA_DICTIONARY.md. Cleaning and cohort rules live in src/clean.py.

# apr_drg_code: The All Patients Refined Diagnosis Related Groups (APR-DRG) Classification Code.
# apr_drg_description: he APR-DRG Classification Code Description.

In [88]:
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from src.clean import build_cohort, clean

DATASET_ID = "46xm-urtu"  # Hospital Inpatient Discharges (SPARCS De-Identified): 2023
APR_DRG = "720"           # Septicemia & disseminated infections (194 = heart failure)
DATA = Path("data"); DATA.mkdir(exist_ok=True)
FIGS = Path("figs"); FIGS.mkdir(exist_ok=True)
RAW_PATH = DATA / f"sparcs_2023_drg{APR_DRG}.parquet"

In [89]:
if not RAW_PATH.exists():
    # ~131k sepsis discharges in 2023; one request, takes a minute or two.
    url = (
        f"https://health.data.ny.gov/resource/{DATASET_ID}.csv"
        f"?apr_drg_code={APR_DRG}&$limit=500000"
    )
    raw = pd.read_csv(url, dtype=str)
    raw.to_parquet(RAW_PATH, index=False)
raw = pd.read_parquet(RAW_PATH)
print(raw.shape)
print(sorted(raw.columns))

(131139, 33)
['age_group', 'apr_drg_code', 'apr_drg_description', 'apr_mdc_code', 'apr_mdc_description', 'apr_medical_surgical', 'apr_risk_of_mortality', 'apr_severity_of_illness', 'apr_severity_of_illness_code', 'birth_weight', 'ccsr_diagnosis_code', 'ccsr_diagnosis_description', 'ccsr_procedure_code', 'ccsr_procedure_description', 'discharge_year', 'emergency_department_indicator', 'ethnicity', 'facility_name', 'gender', 'hospital_county', 'hospital_service_area', 'length_of_stay', 'operating_certificate_number', 'patient_disposition', 'payment_typology_1', 'payment_typology_2', 'payment_typology_3', 'permanent_facility_id', 'race', 'total_charges', 'total_costs', 'type_of_admission', 'zip_code_3_digits']


In [90]:
pd.set_option('display.max_columns', None)

In [91]:
raw.columns

Index(['hospital_service_area', 'hospital_county',
       'operating_certificate_number', 'permanent_facility_id',
       'facility_name', 'age_group', 'zip_code_3_digits', 'gender', 'race',
       'ethnicity', 'length_of_stay', 'type_of_admission',
       'patient_disposition', 'discharge_year', 'ccsr_diagnosis_code',
       'ccsr_diagnosis_description', 'ccsr_procedure_code',
       'ccsr_procedure_description', 'apr_drg_code', 'apr_drg_description',
       'apr_mdc_code', 'apr_mdc_description', 'apr_severity_of_illness_code',
       'apr_severity_of_illness', 'apr_risk_of_mortality',
       'apr_medical_surgical', 'payment_typology_1', 'payment_typology_2',
       'payment_typology_3', 'birth_weight', 'emergency_department_indicator',
       'total_charges', 'total_costs'],
      dtype='str')

In [92]:
def column_stats(df: pd.DataFrame, col: str, top: int = 15) -> None:
    # Column object type
    s = df[col]
    print(f"=== {col} | dtype: {s.dtype}")

    # max/min
    print(f"max = {max(s)}, min = {min(s)}")

    # Missingness
    print(f"missing: {s.isna().sum():,} ({s.isna().mean():.1%})")

    # Unique values and how often each appears
    print(f"unique values: {s.nunique():,}")
    print(list(s.unique()))
    counts = s.value_counts(dropna=False)
    print(pd.DataFrame({"n": counts, "percent": (counts / len(s) * 100).round(2)}).head(top))

    # Is this really a number stored as text? Show the values that won't convert.
    as_num = pd.to_numeric(s.str.replace(",", ""), errors="coerce")
    if as_num.notna().mean() > 0.5:
        not_num = s[as_num.isna() & s.notna()]
        print("mostly numeric. values that don't convert:", not_num.value_counts().to_dict())
        print(as_num.describe().round(2).to_dict())
    print()

    # Outliers with bar plot if the column is numeric



In [93]:
column_stats(raw, 'total_costs')

=== total_costs | dtype: str
max = 99,989.00, min = 1,002.99
missing: 0 (0.0%)
unique values: 129,240
['11,201.40', '10,249.28', '113,186.20', '6,271.71', '20,721.17', '3,856.41', '5,496.58', '55,964.04', '41,395.31', '29,628.81', '18,138.28', '4,244.42', '9,589.07', '13,304.93', '83,717.32', '11,962.02', '50,004.25', '33,215.32', '34,668.29', '26,584.51', '21,827.32', '9,247.53', '11,745.23', '9,990.50', '15,644.92', '318,111.20', '8,663.11', '38,798.87', '23,168.28', '197,917.79', '10,414.53', '21,581.02', '23,575.20', '7,913.12', '13,210.52', '12,074.91', '7,736.15', '32,065.42', '20,323.72', '129,430.80', '49,199.59', '18,740.00', '42,888.94', '33,489.96', '70,665.33', '49,878.99', '5,740.04', '66,947.41', '373,555.83', '15,132.37', '16,607.26', '18,828.19', '28,884.15', '7,821.91', '32,071.00', '37,301.71', '7,002.19', '18,636.27', '27,515.92', '17,109.15', '9,877.71', '24,868.02', '29,192.69', '76,096.64', '6,825.94', '72,062.03', '14,091.04', '28,310.95', '28,763.88', '25,345.97

In [10]:
raw.describe()

,hospital_service_area,hospital_county,operating_certificate_number,permanent_facility_id,facility_name,age_group,zip_code_3_digits,gender,race,ethnicity,length_of_stay,type_of_admission,patient_disposition,discharge_year,ccsr_diagnosis_code,ccsr_diagnosis_description,ccsr_procedure_code,ccsr_procedure_description,apr_drg_code,apr_drg_description,apr_mdc_code,apr_mdc_description,apr_severity_of_illness_code,apr_severity_of_illness,apr_risk_of_mortality,apr_medical_surgical,payment_typology_1,payment_typology_2,payment_typology_3,birth_weight,emergency_department_indicator,total_charges,total_costs
count,131046,131046,131046,131046,131139,131139,129839,131139,131139,131139,131139,131139,131139,131139,131139,131139,85047,85047,131139,131139,131139,131139,131139,131139,131139,131139,131139,77922,20502,193,131139,131139,131139
unique,8,56,153,187,188,5,50,3,4,4,114,6,19,1,6,6,191,191,1,1,1,1,4,4,4,1,9,9,9,35,2,130459,129240
top,New York City,Manhattan,7002054,001637,NewYork-Presbyterian/Queens,70 or Older,112,M,White,Not Span/Hispanic,3,Emergency,Home or Self Care,2023,INF002,Septicemia,ADM015,ADMINISTRATION OF ANTIBIOTICS,720,SEPTICEMIA AND DISSEMINATED INFECTIONS,18,"INFECTIOUS & PARASITIC DISEASES, SYSTEMIC OR U...",4,Extreme,Extreme,Medical,Medicare,Medicaid,Self-Pay,03100,Y,"12,597.00","6,280.75"
freq,50875,16083,6956,2838,2838,68921,13196,65613,78423,99819,16256,125490,53209,131139,130377,130377,10836,10836,131139,131139,131139,131139,55204,55204,58303,131139,84734,32871,9107,18,124538,4,3


### Data Cleaning

In [ ]:
df, cohort_steps = build_cohort(clean(raw))
print(cohort_steps)
print(f"Cohort: {len(df):,} discharges, {df.permanent_facility_id.nunique()} hospitals, "
      f"mortality {df.died.mean():.1%}, died or hospice {df.died_or_hospice.mean():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.groupby("apr_risk_of_mortality")["died"].mean().sort_values().plot.bar(ax=axes[0], title="Mortality by APR risk of mortality")
df["length_of_stay"].clip(upper=40).plot.hist(bins=40, ax=axes[1], title="Length of stay (clipped at 40)")
df.groupby("permanent_facility_id").size().plot.hist(bins=40, ax=axes[2], title="Sepsis discharges per hospital")
plt.tight_layout(); plt.savefig(FIGS / "eda.png", dpi=150)

print(df.groupby("apr_severity_of_illness_code").agg(n=("died", "size"), mortality=("died", "mean"),
                                                     median_cost=("total_costs", "median"),
                                                     median_los=("length_of_stay", "median")))

In [ ]:
# Hospital is NOT a feature. Race is excluded here and used for the fairness check.
BASE_FEATURES = ["age_group", "gender", "type_of_admission",
                 "emergency_department_indicator", "apr_severity_of_illness_code"]
FEATURE_SETS = {
    "full": BASE_FEATURES + ["apr_risk_of_mortality"],
    "no_rom": BASE_FEATURES,  # sensitivity: ROM can absorb complications
    "full_plus_payer": BASE_FEATURES + ["apr_risk_of_mortality", "payment_typology_1"],
}
y = df["died"].to_numpy()
groups = df["permanent_facility_id"].to_numpy()
gkf = GroupKFold(n_splits=5)

def oof_predict(make_model, X, target=None):
    target = y if target is None else target
    pred = np.zeros(len(X))
    for tr, te in gkf.split(X, target, groups):
        m = make_model().fit(X.iloc[tr], target[tr])
        pred[te] = m.predict_proba(X.iloc[te])[:, 1]
    return pred

def logreg(cols):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cols)])
    return lambda: Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=2000))])

def lgbm():
    return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=31,
                              min_child_samples=50, verbose=-1)

preds, rows = {}, []
for name, cols in FEATURE_SETS.items():
    X = df[cols].fillna("Unknown").astype(str)
    preds[f"lr_{name}"] = oof_predict(logreg(cols), X)
    if name == "full":
        Xc = X.astype("category")
        preds["lgbm_full"] = oof_predict(lgbm, Xc)

for name, p in preds.items():
    rows.append({"model": name, "auroc": roc_auc_score(y, p),
                 "auprc": average_precision_score(y, p), "brier": brier_score_loss(y, p)})
metrics = pd.DataFrame(rows).round(4)
print(metrics)
metrics.to_csv(FIGS / "model_metrics.csv", index=False)

In [ ]:
plt.figure(figsize=(5, 5))
for name in ["lr_full", "lgbm_full"]:
    frac, mean = calibration_curve(y, preds[name], n_bins=10, strategy="quantile")
    plt.plot(mean, frac, marker="o", label=name)
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("Predicted mortality"); plt.ylabel("Observed mortality"); plt.legend()
plt.title("Calibration (out-of-fold)"); plt.tight_layout()
plt.savefig(FIGS / "calibration.png", dpi=150)

In [ ]:
def hospital_oe(p, outcome="died"):
    h = (df.assign(p=p, o=df[outcome]).groupby(["permanent_facility_id", "facility_name"])
           .agg(n=("o", "size"), observed=("o", "sum"), expected=("p", "sum"))
           .reset_index())
    h["oe"] = h["observed"] / h["expected"]
    e = h["expected"].to_numpy()[:, None]
    lo95, hi95 = stats.poisson.ppf([0.025, 0.975], e).T
    lo998, hi998 = stats.poisson.ppf([0.001, 0.999], e).T
    h["flag"] = np.select(
        [h.observed > hi998, h.observed < lo998, h.observed > hi95, h.observed < lo95],
        ["high (99.8%)", "low (99.8%)", "high (95%)", "low (95%)"], default="within")
    return h

hosp = hospital_oe(preds["lr_full"])
print(hosp["flag"].value_counts())
hosp.sort_values("oe", ascending=False).to_csv(FIGS / "hospital_oe.csv", index=False)

e = np.linspace(max(hosp.expected.min(), 0.5), hosp.expected.max(), 300)
plt.figure(figsize=(8, 5))
plt.scatter(hosp.expected, hosp.oe, s=14, alpha=0.7)
for (a, b), style in [((0.025, 0.975), "--"), ((0.001, 0.999), ":")]:
    plt.plot(e, stats.poisson.ppf(a, e) / e, "k" + style, lw=1)
    plt.plot(e, stats.poisson.ppf(b, e) / e, "k" + style, lw=1)
plt.axhline(1, color="k", lw=0.8)
plt.xscale("log"); plt.xlabel("Expected deaths (log scale)"); plt.ylabel("Observed / expected")
plt.title(f"Sepsis mortality O/E by hospital, NY 2023 (APR-DRG {APR_DRG})")
plt.tight_layout(); plt.savefig(FIGS / "funnel_plot.png", dpi=150)

In [ ]:
ranks = pd.DataFrame({k: hospital_oe(preds[f"lr_{k}"]).set_index("permanent_facility_id")["oe"]
                      for k in FEATURE_SETS})
y_hosp = df["died_or_hospice"].to_numpy()
cols = FEATURE_SETS["full"]
p_hosp = oof_predict(logreg(cols), df[cols].fillna("Unknown").astype(str), target=y_hosp)
ranks["died_or_hospice"] = hospital_oe(p_hosp, "died_or_hospice").set_index("permanent_facility_id")["oe"]
print(ranks.corr(method="spearman").round(3))

# Which hospitals change flag status when hospice counts as death?
flags = pd.DataFrame({
    "died": hospital_oe(preds["lr_full"]).set_index("permanent_facility_id")["flag"],
    "died_or_hospice": hospital_oe(p_hosp, "died_or_hospice").set_index("permanent_facility_id")["flag"],
})
print(pd.crosstab(flags["died"], flags["died_or_hospice"]))

In [ ]:
fair = (df.assign(p=preds["lr_full"]).groupby("race")
          .agg(n=("died", "size"), observed=("died", "mean"), expected=("p", "mean")))
fair["oe"] = fair["observed"] / fair["expected"]
print(fair.round(3))